In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from scipy import stats
import x13_arima_analysis as x13
import openpyxl

In [ ]:

file_path = "RawData/Import_index.xlsx"
data = pd.read_excel(file_path, sheet_name="Data")

data["Date"] = pd.to_datetime(data["Date"])
data = data.sort_values("Date")

data["Quarter"] = data["Date"].dt.to_period("Q")
quarterly_data = data.groupby("Quarter").mean(numeric_only=True).reset_index()

print(quarterly_data.head())

In [ ]:
countries = [col for col in quarterly_data.columns if col != "Quarter"]
adjusted_data = quarterly_data[["Quarter"]].copy()

for country in countries:
    print(f"\nTest sezónnosti a očištění pro: {country}")

    ts_data = pd.Series(
        quarterly_data[country].values,
        index=quarterly_data["Quarter"].dt.to_timestamp()
    )

    try:
        stl = STL(ts_data, period=4).fit()
        remainder = stl.resid
        seasonal = stl.seasonal
        seasonal_strength = max(0, 1 - np.var(remainder) / np.var(remainder + seasonal))
        print("Síla sezónnosti (STL):", seasonal_strength)
    except Exception as e:
        print("STL dekompozice selhala:", e)
        seasonal_strength = 0

    # ADF test
    adf_p = adfuller(ts_data.dropna())[1]
    print("ADF p-hodnota:", adf_p)

    ocsb_p = 1.0

    if seasonal_strength > 0.64 or adf_p > 0.05 or ocsb_p < 0.05:
        print("Data vykazují sezónnost/nestacionaritu - provádím očištění.")

        try:
            seas = x13.x13_arima_analysis(ts_data, x12path=None)
            adjusted_series = seas.seasadj
            adjusted_data[f"{country}_adjusted"] = adjusted_series.values
        except Exception as e:
            print("Sezónní očištění selhalo:", e)
            adjusted_data[f"{country}_adjusted"] = ts_data.values
    else:
        print("Data nevyžadují sezónní očištění - používám původní.")
        adjusted_data[f"{country}_adjusted"] = ts_data.values

print(adjusted_data.head())

In [ ]:
with pd.ExcelWriter(file_path, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
    adjusted_data.to_excel(writer, sheet_name="Adjusted_Data", index=False)